###Moving data to bronze layer

In [0]:
import pyspark.sql.functions as F

dbutils.widgets.text("volume_path", "/Volumes/dbr_dev_ua5816bd/gulidov_artem_staging/raw_csv/products_raw.csv.gz")
dbutils.widgets.text("bronze_table_path", "dbr_dev_ua5816bd.gulidov_artem_bronze.products_bronze")

df = spark.read \
    .option("header", True) \
    .option("delimiter", "\t") \
    .option("multiLine", True) \
    .option("escape", "\"") \
    .option("compression", "gzip") \
    .csv(dbutils.widgets.get("volume_path"))

df = df.withColumn("ingested_at", F.current_timestamp())

(
    df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(dbutils.widgets.get("bronze_table_path"))
)